kaggle環境で実行したものです。

In [ ]:
import subprocess
import sys

print("="*60)
print("環境確認")
print("="*60)

# GPU情報
print("\n【GPU情報】")
!nvidia-smi

# Pythonバージョン
print("\n【Pythonバージョン】")
!python --version

# ディスク容量
print("\n【ディスク容量】")
!df -h /kaggle/working

In [ ]:
import sys

print("依存ライブラリをインストール中...")

# 基本的なライブラリ
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# LeRobotのインストール（最新安定版）
!pip install -q "lerobot>=0.3.0"

# 追加ライブラリ
!pip install -q pandas pyarrow pillow matplotlib opencv-python

print("✓ インストール完了")

# バージョン確認
import torch
import lerobot
print(f"\nPyTorch version: {torch.__version__}")
print(f"LeRobot version: {lerobot.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:
import os
from pathlib import Path

print("="*60)
print("データセットの確認")
print("="*60)

# Kaggleにアップロードしたデータセットのパス
# 左側のデータパネルからデータセットを追加し、パスを確認してください
dataset_path = "/kaggle/input/datasets/naruhisahayashi/smolvla-dataset-mtl-3/dataset"  

print(f"\nデータセットパス: {dataset_path}")

# ディレクトリ構造の確認
if os.path.exists(dataset_path):
    print("\n【ディレクトリ構造】")
    for root, dirs, files in os.walk(dataset_path):
        level = root.replace(dataset_path, '').count(os.sep)
        indent = ' ' * 2 * level
        print(f'{indent}{os.path.basename(root)}/')
        subindent = ' ' * 2 * (level + 1)
        for file in files[:5]:  # 最初の5ファイルのみ表示
            print(f'{subindent}{file}')
        if len(files) > 5:
            print(f'{subindent}... and {len(files)-5} more files')
else:
    print(f"⚠️ 警告: {dataset_path} が見つかりません")
    print("左側のデータパネル（+ Add data）からデータセットを追加してください")

In [ ]:
!pip install 'lerobot[dataset]'

In [ ]:
import sys
import importlib.util
import importlib.metadata

print("Python:", sys.executable)
print("av spec:", importlib.util.find_spec("av"))

try:
    import av
    print("av version:", av.__version__)
    print("av location:", av.__file__)
except Exception as e:
    print("av import error:", repr(e))

try:
    print("Installed distribution:", importlib.metadata.version("av"))
except Exception as e:
    print("Distribution error:", repr(e))

In [ ]:
import lerobot.utils.import_utils as iu

print("Before:", iu._require_package_cache)

iu._require_package_cache.clear()

from lerobot.datasets.lerobot_dataset import LeRobotDataset

print("LeRobotDataset のインポート成功！")

In [ ]:
from lerobot.datasets.lerobot_dataset import LeRobotDataset
import json

print("="*60)
print("データセットの基本情報")
print("="*60)

try:
    # ローカルパスからデータセットをロード
    dataset = LeRobotDataset(dataset_path)
    
    print(f"\n✓ データセットの読み込みに成功")
    print(f"総エピソード数: {dataset.num_episodes}")
    print(f"総フレーム数: {dataset.num_frames}")
    print(f"FPS: {dataset.fps}")
    
    # 特徴量の確認
    print(f"\n【特徴量（Features）】")
    for key, value in dataset.features.items():
        print(f"  {key}: {value}")
        
except Exception as e:
    print(f"エラー: {e}")
    print("\n代替方法: 直接parquetファイルを読み込む")
    import pandas as pd
    train_dir = Path(dataset_path) / "train"
    parquet_files = list(train_dir.glob("*.parquet"))
    print(f"Parquetファイル数: {len(parquet_files)}")
    if parquet_files:
        df = pd.read_parquet(parquet_files[0])
        print(f"\n最初のエピソードのフレーム数: {len(df)}")
        print(f"カラム: {list(df.columns)}")

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

print("="*60)
print("データの品質検査")
print("="*60)

train_dir = Path(dataset_path) / "train"
parquet_files = sorted(train_dir.glob("*.parquet"))

print(f"\n検査対象エピソード数: {len(parquet_files)}")

# 統計情報の集計
total_frames = 0
state_dims = []
action_dims = []
tasks = set()
missing_data = {
    "state": 0,
    "action": 0,
    "task": 0,
    "image": 0
}

for i, ep_file in enumerate(parquet_files):
    df = pd.read_parquet(ep_file)
    total_frames += len(df)
    
    # 次元の確認
    if i == 0:
        state_dims.append(len(df["observation.state"].iloc[0]))
        action_dims.append(len(df["action"].iloc[0]))
    
    # 欠損値のチェック
    missing_data["state"] += df["observation.state"].isnull().sum()
    missing_data["action"] += df["action"].isnull().sum()
    missing_data["task"] += df["task"].isnull().sum()
    
    # タスクの収集
    tasks.update(df["task"].unique())

print(f"\n【基本統計】")
print(f"  総フレーム数: {total_frames}")
print(f"  1エピソード平均フレーム数: {total_frames / len(parquet_files):.1f}")
print(f"  State次元: {state_dims[0]}")
print(f"  Action次元: {action_dims[0]}")
print(f"  一意のタスク数: {len(tasks)}")

print(f"\n【欠損値チェック】")
for key, count in missing_data.items():
    status = "✓" if count == 0 else "⚠️"
    print(f"  {status} {key}: {count}個")

if len(tasks) > 0:
    print(f"\n【タスク指示の例（最初の5個）】")
    for i, task in enumerate(list(tasks)[:5], 1):
        print(f"  {i}. {task}")

In [ ]:
import pandas as pd
from pathlib import Path
import json

print("="*60)
print("エピソード統計")
print("="*60)

# tasks.jsonlの読み込み
tasks_file = Path(dataset_path) / "meta" / "tasks.jsonl"
if tasks_file.exists():
    tasks_data = []
    with open(tasks_file, 'r') as f:
        for line in f:
            tasks_data.append(json.loads(line))
    
    print(f"\n【タスク分布】")
    print(f"  総タスク数: {len(tasks_data)}")
    
    # info.jsonの読み込み
    info_file = Path(dataset_path) / "meta" / "info.json"
    if info_file.exists():
        with open(info_file, 'r') as f:
            info = json.load(f)
        
        print(f"\n【データセット情報】")
        print(f"  Codebase version: {info.get('codebase_version', 'N/A')}")
        print(f"  Robot type: {info.get('robot_type', 'N/A')}")
        print(f"  FPS: {info.get('fps', 'N/A')}")
        print(f"  Total episodes: {info.get('total_episodes', 'N/A')}")

# 各エピソードのフレーム数
train_dir = Path(dataset_path) / "train"
parquet_files = sorted(train_dir.glob("*.parquet"))

frame_counts = []
for ep_file in parquet_files:
    df = pd.read_parquet(ep_file)
    frame_counts.append(len(df))

print(f"\n【エピソード長さの統計】")
print(f"  最小: {min(frame_counts)} フレーム")
print(f"  最大: {max(frame_counts)} フレーム")
print(f"  平均: {sum(frame_counts)/len(frame_counts):.1f} フレーム")
print(f"  中央値: {sorted(frame_counts)[len(frame_counts)//2]} フレーム")

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
import numpy as np

print("="*60)
print("サンプルデータの可視化")
print("="*60)

# 最初のエピソードからサンプルを取得
train_dir = Path(dataset_path) / "train"
parquet_files = sorted(train_dir.glob("*.parquet"))

if parquet_files:
    df = pd.read_parquet(parquet_files[0])
    
    # ランダムに3フレーム選択
    sample_indices = np.random.choice(len(df), min(3, len(df)), replace=False)
    
    fig, axes = plt.subplots(1, len(sample_indices), figsize=(15, 5))
    if len(sample_indices) == 1:
        axes = [axes]
    
    for idx, ax in zip(sample_indices, axes):
        row = df.iloc[idx]
        
        # 画像の読み込み
        ep_num = int(row['episode_index'])
        frame_num = int(row['frame_index'])
        img_path = Path(dataset_path) / "train" / "images" / "observation.images.camera1" / f"episode_{ep_num:06d}" / f"frame_{frame_num:06d}.jpg"
        
        if img_path.exists():
            from PIL import Image
            img = Image.open(img_path)
            ax.imshow(img)
            ax.axis('off')
            ax.set_title(f"Frame {frame_num}\n{row['task'][:40]}...", fontsize=8)
    
    plt.tight_layout()
    plt.show()
    
    # StateとActionの可視化
    print("\n【StateとActionの統計（最初のエピソード）】")
    print(f"  State range: [{df['observation.state'].apply(lambda x: np.array(x).min()).min():.3f}, {df['observation.state'].apply(lambda x: np.array(x).max()).max():.3f}]")
    print(f"  Action range: [{df['action'].apply(lambda x: np.array(x).min()).min():.3f}, {df['action'].apply(lambda x: np.array(x).max()).max():.3f}]")
else:
    print("⚠️ Parquetファイルが見つかりません")

In [ ]:
import os

for current, dirs, files in os.walk(dataset_path):
    level = current.replace(dataset_path, "").count(os.sep)

    if level > 2:
        dirs[:] = []
        continue

    print("  " * level + os.path.basename(current) + "/")

    for filename in files[:8]:
        print("  " * (level + 1) + filename)

In [ ]:
import json
import os

dataset_path = (
    "/kaggle/input/datasets/naruhisahayashi/smolvla-dataset-mtl-3/dataset"
)

info_path = os.path.join(dataset_path, "meta", "info.json")

with open(info_path, "r") as f:
    info = json.load(f)

print(json.dumps(info, indent=2, ensure_ascii=False))

In [ ]:
import glob
import pyarrow.parquet as pq

parquet_files = glob.glob(
    os.path.join(dataset_path, "train", "*.parquet")
)

print("Parquet ファイル数:", len(parquet_files))

if parquet_files:
    table = pq.read_table(parquet_files[0])

    print("\n列名:")
    print(table.column_names)

    print("\nスキーマ:")
    print(table.schema)

    print("\n先頭2行:")
    print(table.slice(0, 2).to_pylist())

In [ ]:
import os
import glob
import json
import pyarrow.parquet as pq

dataset_path = (
    "/kaggle/input/datasets/naruhisahayashi/smolvla-dataset-mtl-3/dataset"
)

# info.json の全内容
with open(os.path.join(dataset_path, "meta", "info.json")) as f:
    info = json.load(f)

print("=== info.json のキー ===")
print(list(info.keys()))

print("\n=== Parquet の列とスキーマ ===")
files = sorted(glob.glob(os.path.join(dataset_path, "train", "*.parquet")))
print("ファイル数:", len(files))

table = pq.read_table(files[0])
print(table.schema)
print("\n列名:", table.column_names)

print("\n=== 最初の行（画像データは省略） ===")
row = table.slice(0, 1).to_pylist()[0]

for key, value in row.items():
    if isinstance(value, dict):
        print(key, "->", {
            k: (f"<データ省略: {len(v)} bytes>" if isinstance(v, bytes) else type(v).__name__)
            for k, v in value.items()
        })
    elif isinstance(value, bytes):
        print(key, f"<データ省略: {len(value)} bytes>")
    else:
        print(key, repr(value)[:300])

In [ ]:
tasks_path = os.path.join(dataset_path, "meta", "tasks.jsonl")

with open(tasks_path, encoding="utf-8") as f:
    for line in list(f)[:5]:
        print(line.rstrip())

In [ ]:
import os
from collections import Counter

images_dir = os.path.join(dataset_path, "train", "images")

print("画像フォルダの存在:", os.path.isdir(images_dir))

if os.path.isdir(images_dir):
    all_files = []
    for root, dirs, files in os.walk(images_dir):
        all_files.extend(
            os.path.join(root, f)
            for f in files
        )

    print("画像フォルダ内のファイル数:", len(all_files))
    print("拡張子ごとの件数:",
          Counter(os.path.splitext(f)[1].lower() for f in all_files))

    print("\n先頭20ファイル:")
    for f in sorted(all_files)[:20]:
        print(os.path.relpath(f, images_dir))

In [ ]:
dataset_path = (
    "/kaggle/input/notebooks/naruhisahayashi/convert-lerobot-mtl/train-smolvla-mtl-dataset-lerobot"
)


In [ ]:
# from lerobot import LeRobotDataset
import torch

print("="*60)
print("学習前のデータセット検証")
print("="*60)

try:
    # dataset = LeRobotDataset(dataset_path)
    dataset = LeRobotDataset(
        repo_id="local/train-smolvla-mtl-dataset-lerobot",
        root=dataset_path,
    )
    
    # サンプルの取得
    sample = dataset[0]
    
    print("\n【サンプルデータの詳細】")
    print(f"  observation.images.cam_overhead: {sample['observation.images.camera1'].shape}")
    print(f"  observation.state: {sample['observation.state'].shape}")
    print(f"  action: {sample['action'].shape}")
    print(f"  task: {sample['task']}")
    
    # 画像の正規化確認
    img = sample['observation.images.camera1']
    print(f"\n【画像テンソル情報】")
    print(f"  dtype: {img.dtype}")
    print(f"  min: {img.min():.3f}")
    print(f"  max: {img.max():.3f}")
    print(f"  mean: {img.mean():.3f}")
    
    # GPU利用可能か確認
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"\n  使用デバイス: {device}")
    
    print("\n✓ データセットの準備完了")
    
except Exception as e:
    print(f"⚠️ エラー: {e}")
    print("データセット形式に問題がある可能性があります")

In [ ]:
!pip install num2words

In [ ]:
import subprocess
import os
import re

print("=" * 60)
print("SmolVLA ファインチューニング開始")
print("=" * 60)

output_dir = "/kaggle/working/outputs/train/my_smolvla"
log_path = "/kaggle/working/train.log"

train_cmd = f"""
lerobot-train \
  --policy.path=lerobot/smolvla_base \
  --dataset.repo_id={dataset_path} \
  --rename_map='{{"observation.images.cam_overhead":"observation.images.camera1"}}' \
  --policy.input_features='{{"observation.images.camera1":{{"type":"VISUAL","shape":[3,480,640]}}}}' \
  --policy.empty_cameras=2 \
  --batch_size=8 \
  --steps=5000 \
  --save_freq=500 \
  --output_dir={output_dir} \
  --job_name=my_smolvla_training \
  --policy.device=cuda \
  --policy.push_to_hub=false \
  --wandb.enable=false
"""

print("学習ステップ: 10000")
print("画面表示: 100 stepごと")
print(f"全ログ保存先: {log_path}")
print("\n学習を開始します...\n")

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"

try:
    with open(log_path, "w", buffering=1) as log_file:
        process = subprocess.Popen(
            train_cmd,
            shell=True,
            executable="/bin/bash",
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            env=env,
        )

        last_printed_step = -1

        for line in process.stdout:
            # 全ログをファイルに保存
            log_file.write(line)
            log_file.flush()

            # LeRobotの進捗表示からstep番号を取得
            match = re.search(
                r"(\d+)\s*/\s*5000|step\s*[=:]\s*(\d+)",
                line,
                re.IGNORECASE,
            )

            if match:
                step = int(match.group(1) or match.group(2))

                if step >= 100 and step // 100 > last_printed_step // 100:
                    print(line.rstrip(), flush=True)
                    last_printed_step = step

            # エラー発生時は画面にも表示
            elif re.search(
                r"error|exception|traceback",
                line,
                re.IGNORECASE,
            ):
                print(line.rstrip(), flush=True)

        return_code = process.wait()

    if return_code == 0:
        print("\n学習完了！")
    else:
        print(f"\n学習がエラー終了しました。終了コード: {return_code}")

    print(f"詳細ログ保存先: {log_path}")

except Exception as e:
    print(f"\nエラーが発生しました: {e}")

In [ ]:
import os
from pathlib import Path
import matplotlib.pyplot as plt

print("="*60)
print("学習結果の確認")
print("="*60)

output_dir = Path("/kaggle/working/outputs/train/my_smolvla")

# 生成されたファイルの確認
print("\n【出力ファイル】")
if output_dir.exists():
    for item in output_dir.rglob("*"):
        if item.is_file():
            rel_path = item.relative_to(output_dir)
            print(f"  {rel_path}")
else:
    print("⚠️ 出力ディレクトリが見つかりません")

# チェックポイントの確認
checkpoint_dir = output_dir / "checkpoints" / "last" / "pretrained_model"
if checkpoint_dir.exists():
    print(f"\n✓ チェックポイントが保存されました: {checkpoint_dir}")
    
    # ファイルサイズ
    total_size = sum(f.stat().st_size for f in checkpoint_dir.rglob("*") if f.is_file())
    print(f"  チェックポイントサイズ: {total_size / (1024**2):.1f} MB")
else:
    print("\n⚠️ チェックポイントが見つかりません")

In [ ]:
import torch
from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy
from pathlib import Path

print("="*60)
print("チェックポイントの検証")
print("="*60)

checkpoint_path = Path("/kaggle/working/outputs/train/my_smolvla/checkpoints/last/pretrained_model")

if checkpoint_path.exists():
    try:
        print("モデルをロード中...")
        policy = SmolVLAPolicy.from_pretrained(str(checkpoint_path))
        device = "cuda" if torch.cuda.is_available() else "cpu"
        policy = policy.to(device)
        policy.eval()
        
        print("✓ モデルのロードに成功")
        print(f"  デバイス: {device}")
        print(f"  モデルタイプ: {type(policy)}")
        
        # パラメータ数の確認
        total_params = sum(p.numel() for p in policy.parameters())
        trainable_params = sum(p.numel() for p in policy.parameters() if p.requires_grad)
        
        print(f"  総パラメータ数: {total_params:,}")
        print(f"  学習可能パラメータ: {trainable_params:,}")
        
    except Exception as e:
        print(f"⚠️ エラー: {e}")
        import traceback
        traceback.print_exc()
else:
    print(f"⚠️ チェックポイントパスが存在しません: {checkpoint_path}")

In [ ]:

import torch
from lerobot.datasets.lerobot_dataset import LeRobotDataset
from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy
from pathlib import Path

print("=" * 60)
print("推論テスト")
print("=" * 60)

# データセットとチェックポイント
dataset = LeRobotDataset(dataset_path)

checkpoint_path = Path(
    "/kaggle/working/outputs/train/my_smolvla/checkpoints/last/pretrained_model"
)

if not checkpoint_path.exists():
    raise FileNotFoundError(
        f"チェックポイントが見つかりません: {checkpoint_path}"
    )

# モデル読み込み
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

policy = SmolVLAPolicy.from_pretrained(str(checkpoint_path))
policy = policy.to(device)
policy.eval()

# サンプル観測
sample = dataset[0]

task = sample["task"]
if isinstance(task, (list, tuple)):
    task = task[0]

print("\n【テスト観測】")
print("  Task:", task)
print("  State shape:", sample["observation.state"].shape)
print("  Image shape:", sample["observation.images.camera1"].shape)

# 画像を [B, C, H, W] にする
image = sample["observation.images.camera1"]
if image.ndim == 3:
    image = image.unsqueeze(0)
image = image.to(device=device, dtype=torch.float32)

# 画像が uint8 の場合は [0, 1] に正規化
if image.max() > 1:
    image = image / 255.0

# 状態を [B, state_dim] にする
state = sample["observation.state"]
if state.ndim == 1:
    state = state.unsqueeze(0)
state = state.to(device=device, dtype=torch.float32)

# SmolVLA に付属するトークナイザーを使用
processor = policy.model.vlm_with_expert.processor
encoded = processor.tokenizer(
    [str(task)],
    padding="max_length",
    max_length=policy.config.tokenizer_max_length,
    truncation=True,
    return_tensors="pt",
)

language_tokens = encoded["input_ids"].to(device)
language_mask = encoded["attention_mask"].to(device).bool()

# モデルが期待する入力キー
inference_obs = {
    "observation.images.camera1": image,
    "observation.state": state,
    "observation.language.tokens": language_tokens,
    "observation.language.attention_mask": language_mask,
}

print("\n推論に渡すテンソル:")
for key, value in inference_obs.items():
    print(f"  {key}: shape={tuple(value.shape)}, dtype={value.dtype}")

print("\n推論実行中...")

with torch.no_grad():
    action = policy.select_action(inference_obs)

print("✓ 推論完了")
print("  Action shape:", tuple(action.shape))
print("  Action:", action.detach().cpu().numpy())
print(
    "  Action range: "
    f"[{action.min().item():.3f}, {action.max().item():.3f}]"
)
